# Independent external-validation audit

## tl;dr

This companion audit verifies the machine-readable result produced by SentinelLoop's chronological ULB/Worldline validation. The current result is **share with caveats**: future-test PR-AUC is 0.7656, precision is 0.8727, recall is 0.6486, and false-positive rate is 0.0124%. It supports the scalable detector methodology, not full Red/Blue lifecycle-agent generalization.

## Context & Methods

The source is [OpenML dataset 1597](https://www.openml.org/d/1597). The executable pipeline uses a chronological 60%/20%/20% train/validation/test split. Candidate selection, Platt calibration, and threshold selection use validation only; the future test is opened after those choices are frozen.

### Key Assumptions

- Exact duplicate source rows are removed before splitting because no transaction ID exists.
- The public dataset validates one anonymized card-transaction detector, not Qwen lifecycle reasoning or non-card rails.
- Confidence intervals accompany point estimates because only 74 frauds occur in the future test.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORT_PATH = PROJECT_ROOT / 'data' / 'external_validation' / 'latest.json'
SOURCE_PATH = PROJECT_ROOT / 'data' / 'external' / 'creditcard.parquet'
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert report['methodology']['test_opened_after_selection'] is True
{'report': str(REPORT_PATH), 'source_present': SOURCE_PATH.exists(), 'assessment': report['overall_assessment']}

{'report': '/Users/u367403/Library/CloudStorage/OneDrive-UnitedAirlines/Projects/Hackathon/MasterCard/data/external_validation/latest.json',
 'source_present': True,
 'assessment': 'share_with_caveats'}

## Data

Reconcile the source totals, duplicate remediation, and chronological boundary. The raw Parquet file is optional and intentionally git-ignored; the committed audit result still exposes all reconciliation evidence.

In [2]:
quality = report['data_quality']
remediation = quality['remediation']
assert quality['row_count'] == 284_807
assert quality['fraud_count'] == 492
assert remediation['raw_row_count'] - remediation['modeled_row_count'] == remediation['exact_duplicate_rows_removed']
splits = report['dataset']['split_summary']
assert splits['train']['time_max_seconds'] <= splits['validation']['time_min_seconds']
assert splits['validation']['time_max_seconds'] <= splits['test']['time_min_seconds']
pd.DataFrame([{
    'raw_rows': quality['row_count'],
    'raw_frauds': quality['fraud_count'],
    'duplicates_removed': remediation['exact_duplicate_rows_removed'],
    'modeled_rows': remediation['modeled_row_count'],
    'quality_score': quality['score'],
    'model_input_status': quality['model_input_status'],
}])

,raw_rows,raw_frauds,duplicates_removed,modeled_rows,quality_score,model_input_status
0,284807,492,1081,283726,91.89,passed_after_conservative_deduplication


In [3]:
if SOURCE_PATH.exists():
    source = pd.read_parquet(SOURCE_PATH)
    assert len(source) == quality['row_count']
    source['Class'] = pd.to_numeric(source['Class'], errors='raise')
    assert int(source['Class'].sum()) == quality['fraud_count']
    assert int(source.duplicated().sum()) == remediation['exact_duplicate_rows_removed']
    source_check = 'Raw source independently reconciled.'
else:
    source_check = 'Raw source absent; report-level checks completed.'
source_check

'Raw source independently reconciled.'

## Results

Spot-check the confusion-matrix arithmetic, uncertainty fields, calibration comparison, and temporal slices.

In [4]:
metrics = report['defense']['test_metrics']
cm = metrics['confusion_matrix']
calculated_precision = cm['tp'] / (cm['tp'] + cm['fp'])
calculated_recall = cm['tp'] / (cm['tp'] + cm['fn'])
calculated_fpr = cm['fp'] / (cm['fp'] + cm['tn'])
assert abs(calculated_precision - metrics['precision']) < 1e-6
assert abs(calculated_recall - metrics['recall']) < 1e-6
assert abs(calculated_fpr - metrics['false_positive_rate']) < 1e-6
assert metrics['brier_score'] < metrics['null_brier_score']
assert len(metrics['pr_auc_95ci']) == 2
pd.Series({
    'PR-AUC': metrics['pr_auc'],
    'Precision': metrics['precision'],
    'Recall': metrics['recall'],
    'F1': metrics['f1'],
    'False-positive rate': metrics['false_positive_rate'],
    'Brier skill score': metrics['brier_skill_score'],
    'Expected calibration error': metrics['expected_calibration_error'],
}, name='future_test')

PR-AUC                        0.765572
Precision                     0.872727
Recall                        0.648649
F1                            0.744186
False-positive rate           0.000124
Brier skill score             0.596213
Expected calibration error    0.000163
Name: future_test, dtype: float64

In [5]:
temporal = pd.DataFrame(report['defense']['temporal_slices']).T[
    ['event_count', 'fraud_event_count', 'pr_auc', 'precision', 'recall', 'f1', 'false_positive_rate']
]
assert temporal.loc['late_test', 'pr_auc'] < temporal.loc['early_test', 'pr_auc']
temporal

,event_count,fraud_event_count,pr_auc,precision,recall,f1,false_positive_rate
early_test,28373,52,0.834906,0.972973,0.692308,0.808989,0.000035
late_test,28373,22,0.631488,0.666667,0.545455,0.6,0.000212


## Takeaways

1. The narrow detector claim is independently supported under severe real-world class imbalance.
2. Seven false positives across 56,672 legitimate future payments show a promising operating point, with uncertainty explicitly bounded.
3. Later-window PR-AUC falls from 0.8349 to 0.6315, so drift monitoring and recalibration are release requirements.
4. The test cannot support claims about GenAI attack diversity, entity graphs, lifecycle reasoning, or non-card rails. Institution shadow scoring is the next production gate.